# PKG regular payments: do the expectations hold?

**PNC Treasury Management · DSI · Payment Knowledge Graph**

Customer metrics v4 judges every regular stream against what its own history predicts: the next payment around the
usual gap after the last one, an amount near the median of recent payments, and an expected dollar total for the
month (median × payments expected in the month). This notebook backtests those predictions on past data before we
build a separate periodicity table and use them for liquidity and contagion work. Every prediction uses only the
payments **before** it, with the same rules as v4.

**Questions**
1. **Timing:** how often does the next payment arrive on time, early, late, skipped, or never (stream stopped)? How
   many days off is it?
2. **Amount:** how close is the next payment to the median? Is the median the best predictor (against the last
   payment, the mean, the median of the last 3)? Does the spread of the history (MAD) tell us how far to trust it, and
   does the p10–p90 band hold about 80% of next payments?
3. **Month:** how close are received dollars to the expected dollars per stream and per customer, month by month?
   v4's method against payment dates projected from the last payment (× median or × mean), with the shortfall of
   streams that stopped for good shown apart from forecasting error.
4. **Intervals:** do the prediction intervals (next amount, next date, month total) hold what they claim?

Every answer is split by cadence (v4 groups: daily, weekly, biweekly, monthly, quarterly), direction and side
(`in ext` external payer → customer, `out ext` customer → external payee, `in int` / `out int` book to book between
PNC customers, seen from each side) and rail. Own-account transfers are left out, as in v4.

**Rules (v4).** History = the latest `REG_MAX_DAYS` payment days within `REG_WINDOW` months. Regular = at least
`MIN_PAYMENTS` payment days, a known cadence from the median gap and ≥ `REG_MIN` of gaps on time (within
`max(TOL_MIN, TOL_FRAC × median gap)` days; daily: the next business day). A gap ≥ `SKIP_MULT ×` median = skipped;
no payment for that long (daily: `DAILY_STOP_BD` business days) = stopped. The first `WINDOW_MONTHS − EVAL_MONTHS`
months only build history; predictions are judged in the last `EVAL_MONTHS` months.

**Sections:** §1 stream days · §2 predictions · §3 timing · §4 amount · §5 expected dollars per month · §6 figures
(PNG files in `FIG_DIR`, including example streams over time, the three monthly methods and interval coverage) ·
§7 housekeeping.

**Run order:** after `pkg_edges_refresh.ipynb` and `pkg_reference_data.ipynb`. Reads only; writes scratch files.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
SHUFFLE_PARTITIONS = 2000

# ---- Window and sample (a change rebuilds §1 and §2)
WINDOW_END    = None      # last month analysed, e.g. "2026-10"; None = latest month published by pkg_edges_refresh
WINDOW_MONTHS = 24        # months read: the first WINDOW_MONTHS − EVAL_MONTHS only build history
EVAL_MONTHS   = 12        # last months whose payments and months are judged
SAMPLE_PCT    = 2         # % of customers (hash of mdm_id) whose streams are analysed; 100 = every customer
FC_PARTITIONS = 4000      # shuffle partitions for the §2 build only (no effect on the numbers); raise if executors are lost

# ---- v4 rules (a change rebuilds §2); keep equal to pkg_customer_metrics_v4
REG_WINDOW, REG_MAX_DAYS = 24, 26   # history: latest REG_MAX_DAYS payment days within REG_WINDOW months
MIN_PAYMENTS, REG_MIN = 4, 0.70
TOL_FRAC, TOL_MIN = 0.15, 1         # on time: |gap − median gap| ≤ max(TOL_MIN, TOL_FRAC × median gap) days
SKIP_MULT     = 1.75                # a gap ≥ SKIP_MULT × median gap = skipped; silent that long = stopped
DAILY_STOP_BD = 5                   # daily streams: stopped after this many business days without a payment
FIXED_PCT     = 0.01                # v4 amount-low rule: also below median × (1 − FIXED_PCT)
CADENCES = [("weekly", 6, 8), ("biweekly", 12, 17), ("monthly", 26, 35), ("quarterly", 55, 66), ("quarterly", 84, 98)]
CAD_GROUPS = ["daily", "weekly", "biweekly", "monthly", "quarterly"]

# ---- Figures and scratch
FIG_DIR             = "figures/pkg_regular_forecast_check"   # PNG files, next to the notebook (one subfolder per window end)
EXAMPLES_PER_CAD    = 3       # §6: example streams drawn per cadence (time series)
KEEP_SCRATCH        = True    # keep §1/§2 scratch so §3–§6 can be rerun without rebuilding
CLEAN_OTHER_SCRATCH = False   # §7: delete scratch folders of other settings

## Setup

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
# Compatibility shims: Cloudera SPARK3 PySpark is older than the NumPy / pandas in the conda env.
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)

import calendar, hashlib, json, time, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder.appName("pkg_regular_forecast_check")
         .config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
         .config("spark.executor.memory", "8g").config("spark.driver.memory", "4g")
         .config("spark.driver.maxResultSize", "4g").config("spark.executor.memoryOverhead", "3g")
         .config("spark.executor.cores", "2")
         .config("spark.dynamicAllocation.enabled", "true").config("spark.dynamicAllocation.maxExecutors", "512")
         .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
         .config("spark.sql.execution.arrow.pyspark.enabled", "true")
         .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
         .config("spark.ui.showConsoleProgress", "false")
         .enableHiveSupport().getOrCreate())
spark.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
spark.conf.set("spark.sql.ansi.enabled", "false")      # ratios with a zero denominator are NULL, not an error
spark.sparkContext.setLogLevel("ERROR")
# The cluster's Atlas lineage hook can't reach its Kafka topic and prints the full query plan on every write; netty /
# TransportResponseHandler print harmless errors when executors are released. Their loggers are switched off.
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty")
try:
    _L = spark._jvm.org.apache.logging.log4j
    for _n in _QUIET:
        _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
except Exception:
    try:
        _L1 = spark._jvm.org.apache.log4j
        for _n in _QUIET:
            _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
    except Exception as _e:
        print("could not silence the Atlas logs:", str(_e)[:120])
pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- one consolidated HTML output per cell
def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.0f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text):
    return f"<p style='margin:4px 0; font-size:13px'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

# ---- HDFS (Hadoop FileSystem API through the JVM)
HCONF = spark._jsc.hadoopConfiguration()
JPath = spark._jvm.org.apache.hadoop.fs.Path
fs_for = lambda p: JPath(p).getFileSystem(HCONF)
exists = lambda p: fs_for(p).exists(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0
def ls_dirs(p):
    if not exists(p):
        return []
    return [s.getPath().getName() for s in fs_for(p).listStatus(JPath(p)) if s.isDirectory()]

TIMES = {}
class timed:
    def __init__(self, name): self.name = name
    def __enter__(self): self.t0 = time.time()
    def __exit__(self, *a): TIMES[self.name] = (time.time() - self.t0) / 60

def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"
def month_range(a, b):
    out_, m = [], a
    while m <= b:
        out_.append(m); m = month_add(m, 1)
    return out_

def us_bank_holidays(y0, y1):
    """US Federal Reserve holidays (ACH / Fedwire closed). A holiday on a Sunday is observed on Monday; one on a
    Saturday is not observed (the Fed stays open on Friday)."""
    def nth(y, m, wd, k):                      # k-th weekday wd (Mon = 0) of the month; k = -1 → last
        if k > 0:
            d = dt.date(y, m, 1)
            d += dt.timedelta(days=(wd - d.weekday()) % 7 + 7 * (k - 1))
        else:
            d = dt.date(y, m, calendar.monthrange(y, m)[1])
            d -= dt.timedelta(days=(d.weekday() - wd) % 7)
        return d
    out_ = []
    for y in range(y0, y1 + 1):
        fixed = [dt.date(y, 1, 1), dt.date(y, 6, 19), dt.date(y, 7, 4), dt.date(y, 11, 11), dt.date(y, 12, 25)]
        moving = [nth(y, 1, 0, 3), nth(y, 2, 0, 3), nth(y, 5, 0, -1), nth(y, 9, 0, 1), nth(y, 10, 0, 2), nth(y, 11, 3, 4)]
        for d in fixed:
            if d.weekday() == 6:
                d += dt.timedelta(days=1)
            if d.weekday() < 5:
                out_.append(d)
        out_ += moving
    return sorted(out_)
HOLIDAYS = us_bank_holidays(2023, 2028)

def bidx(c):
    """Business-day index of a date column: each business day (Monday–Friday, not a Fed holiday) counts 1; weekends
    and holidays share the next business day's index. So Friday → Monday is 1 business day, Friday → Tuesday after a
    Monday holiday is also 1, and a payment made on a weekend or holiday is on time for a daily stream."""
    n = F.datediff(c, F.to_date(F.lit("1970-01-05")))           # 1970-01-05 is a Monday
    wk = F.floor(n / 7)
    hol = sum(F.when(c > F.to_date(F.lit(h.isoformat())), 1).otherwise(0) for h in HOLIDAYS)
    return (wk * 5 + F.least(n - wk * 7, F.lit(5)) - hol).cast("long")

EPOCH = dt.date(1970, 1, 1)
MONDAY0 = dt.date(1970, 1, 5)
def bidx_py(d):
    n = (d - MONDAY0).days
    return n // 7 * 5 + min(n % 7, 5) - sum(1 for h in HOLIDAYS if d > h)
def month_first(m):
    return dt.date(int(m[:4]), int(m[5:7]), 1)
def month_last(m):
    d = month_first(m) + dt.timedelta(days=32)
    return d - dt.timedelta(days=d.day)
def business_days(m):
    d, n, end = month_first(m), 0, month_last(m)
    while d <= end:
        n += d.weekday() < 5 and d not in HOLIDAYS
        d += dt.timedelta(days=1)
    return n

print(f"Spark {spark.version} | pandas {pd.__version__} | numpy {np.__version__}")

## §1 Stream days

Same as `pkg_payment_regularity` §1 (shared scratch folder: reused when that notebook ran with the same window and
sample): one row per stream and payment day, external counterparties at `ent_l2`, the rail of the day's largest
payment.

In [ ]:
# ---- §1 Stream days -----------------------------------------------------------------------------
T_EDGE_D, T_MAP, T_CUST = f"{DB}.pkg_edge_daily", f"{DB}.pkg_cpty_entity_map_v2", f"{DB}.pkg_cust_dim"
DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")

PROBLEMS = []
NEED = {T_EDGE_D: ["txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "category", "is_self", "amount", "n_txn",
                   "month"],
        T_MAP: ["cpty_id", "ent_l2", "name_class"],
        T_CUST: ["mdm_id", "party_type", "is_tm"]}
if WINDOW_MONTHS < REG_WINDOW // 2 + EVAL_MONTHS:
    PROBLEMS.append({"problem": "window too short", "detail": f"WINDOW_MONTHS {WINDOW_MONTHS} < {REG_WINDOW // 2} months of history + EVAL_MONTHS {EVAL_MONTHS}"})
for t, cols in NEED.items():
    have = set(spark.table(t).columns)
    PROBLEMS += [{"problem": "column missing", "detail": f"{t}.{c}"} for c in cols if c not in have]
_loc = table_location(T_EDGE_D)
PARTS = sorted(r[0].split("=", 1)[1] for r in spark.sql(f"SHOW PARTITIONS {T_EDGE_D}").collect())
PUBLISHED = [m for m in PARTS if exists(f"{_loc}/month={m}/_PKG_BUILD")]
END = WINDOW_END or (PUBLISHED[-1] if PUBLISHED else None)
MONTHS = month_range(month_add(END, -(WINDOW_MONTHS - 1)), END) if END else []
_unpub = [m for m in MONTHS if m not in PUBLISHED]
if _unpub:
    PROBLEMS.append({"problem": "months not published by pkg_edges_refresh", "detail": ", ".join(_unpub)})
if PROBLEMS:
    out(sec(f"§1 stopped: {len(PROBLEMS)} problem(s)", tbl(pd.DataFrame(PROBLEMS))))
    raise AssertionError(f"§1: {len(PROBLEMS)} problem(s), see the table above")

W_START = dt.date.fromisoformat(MONTHS[0] + "-01")
W_END = dt.date(int(END[:4]), int(END[5:7]), calendar.monthrange(int(END[:4]), int(END[5:7]))[1])
VERSION = "reg_v2"          # v2: business days skip Fed holidays
SIG_SD = hashlib.md5(json.dumps({"v": VERSION, "m": MONTHS, "s": SAMPLE_PCT}).encode()).hexdigest()[:10]
SCR = f"{DB_LOC}/_tmp_pkg_regularity"
P_SD = f"{SCR}/sd_{SIG_SD}"
RAIL = {"1": "ACH", "2": "ACH", "3": "ACH", "4": "WIRE", "5": "RTP_PRT", "6": "RTP_P2P", "7": "CHECK", "9": "PCARD"}
K = ["payer", "payer_kind", "payee", "payee_kind", "flow"]

def sampled(c):
    return F.lit(True) if SAMPLE_PCT >= 100 else F.expr(f"pmod(xxhash64({c}), 100) < {int(SAMPLE_PCT)}")

sd_built = not exists(f"{P_SD}/_SUCCESS")
if sd_built:
    with timed("§1 stream days"):
        src_c, dst_c = F.col("src_kind") == "CUST", F.col("dst_kind") == "CUST"
        e = (spark.table(T_EDGE_D).where(F.col("month").isin(MONTHS))
             .withColumn("payer_s", src_c & sampled("src_id"))
             .withColumn("payee_s", dst_c & sampled("dst_id"))
             .where(F.col("payer_s") | F.col("payee_s"))
             .withColumn("flow", F.when(src_c & dst_c & F.coalesce(F.col("is_self"), F.lit(False)), "self")
                                  .when(src_c & dst_c, "book").when(dst_c, "in").when(src_c, "out").otherwise("other"))
             .withColumn("rail", F.create_map(*[F.lit(x) for kv in RAIL.items() for x in kv])[F.substring("category", 1, 1)])
             .withColumn("cp_id", F.when(~src_c, F.col("src_id")).when(~dst_c, F.col("dst_id")))
             .select("txn_date", "src_id", "src_kind", "dst_id", "dst_kind", "flow", "rail", "cp_id", "payer_s", "payee_s",
                     F.col("amount").cast("double").alias("amount"), F.col("n_txn").cast("long").alias("n_txn")))
        mp = spark.table(T_MAP).select(F.col("cpty_id").alias("cp_id"), F.col("ent_l2").alias("cp_ent"),
                                       F.col("name_class").alias("cp_class"))
        # internal rows have no counterparty id: they skip the map join (NULL keys would all land in one task)
        e = (e.where(F.col("cp_id").isNotNull()).join(mp, "cp_id", "left")
              .unionByName(e.where(F.col("cp_id").isNull())
                            .withColumn("cp_ent", F.lit(None).cast("string"))
                            .withColumn("cp_class", F.lit(None).cast("string"))))
        e = (e.withColumn("payer", F.when(F.col("src_kind") == "CPTY", F.coalesce("cp_ent", "src_id")).otherwise(F.col("src_id")))
              .withColumn("payee", F.when(F.col("dst_kind") == "CPTY", F.coalesce("cp_ent", "dst_id")).otherwise(F.col("dst_id")))
              .withColumnRenamed("src_kind", "payer_kind").withColumnRenamed("dst_kind", "payee_kind"))
        sd = e.groupBy(*K, "txn_date").agg(
            F.sum("amount").alias("amt"), F.sum("n_txn").alias("n_txn"),
            F.max(F.struct("amount", "rail")).getField("rail").alias("rail"),
            F.max(F.struct("amount", "cp_class")).getField("cp_class").alias("cp_class"),
            F.max("payer_s").alias("payer_s"), F.max("payee_s").alias("payee_s"))
        w = Window.partitionBy("payer", "payee").orderBy("txn_date")
        sd = (sd.withColumn("prev_dt", F.lag("txn_date").over(w))
                .withColumn("prev_amt", F.lag("amt").over(w))
                .withColumn("gap", F.datediff("txn_date", "prev_dt"))
                .withColumn("gap_bd", bidx(F.col("txn_date")) - bidx(F.col("prev_dt"))))
        sd.write.mode("overwrite").parquet(P_SD)

SD = spark.read.parquet(P_SD)
s1 = (SD.groupBy("flow").agg(F.count("*").alias("payment_days"), F.sum("n_txn").alias("transactions"),
                             (F.sum("amt") / 1e9).alias("dollars_B"))
        .toPandas().sort_values("dollars_B", ascending=False))
_S1_OK = True
out(sec(f"§1 Stream days {MONTHS[0]} → {MONTHS[-1]}, {SAMPLE_PCT}% of customers — "
        + (f"built ({TIMES['§1 stream days']:.1f} min)" if sd_built else "reused"), tbl(s1, dec=("dollars_B",))),
    para(f"Window {W_START} → {W_END}. Scratch {P_SD}."))

## §2 Predictions

For streams with at least `MIN_PAYMENTS` payment days, three kinds of rows get the stream's history at that point:
- **payment:** each payment day in the judged months, with the prediction made after the previous payment;
- **month start:** the first day of each judged month, with the state at the end of the previous month (what v4
  expects for the month);
- **window end:** the day after the window, to catch streams that stopped (no next payment).

From the history: median gap, cadence, regular or not, median amount, p10 / p90, mean, last payment, median of the
last 3, and the spread MAD (median absolute difference from the median). Only regular histories are judged.

In [ ]:
# ---- §2 Predictions -----------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

FC_VERSION = "fc_v3"          # v3: projected payment dates, stops for good, prediction intervals
SIG_FC = hashlib.md5(json.dumps({"v": FC_VERSION, "sd": SIG_SD, "e": EVAL_MONTHS, "r": [REG_WINDOW, REG_MAX_DAYS, MIN_PAYMENTS,
                     REG_MIN, TOL_FRAC, TOL_MIN, SKIP_MULT, DAILY_STOP_BD], "c": CADENCES}).encode()).hexdigest()[:10]
P_PR, P_MO = f"{SCR}/fcp_{SIG_FC}", f"{SCR}/fcm_{SIG_FC}"
EVAL = MONTHS[-EVAL_MONTHS:]
E0 = month_first(EVAL[0])
H_ROWS = REG_MAX_DAYS + EVAL_MONTHS + 2           # rows looked back: payment days plus month-start rows in between
SK = ["payer", "payee", "flow"]
BD_IN = {m: business_days(m) for m in EVAL}

def _gaps(x):
    return (f"if(size({x}) >= 2, zip_with(slice({x}, 2, size({x}) - 1), slice({x}, 1, size({x}) - 1), (u, v) -> u - v), "
            f"cast(array() as array<int>))")
def _nth(x, q):
    """q-quantile of an array (lower element of the sorted array, as in v4); NULL when empty."""
    return f"if(size({x}) = 0, null, element_at(array_sort({x}), cast(floor((size({x}) - 1) * {q}) as int) + 1))"

from contextlib import contextmanager
@contextmanager
def many_partitions(n):
    """Small tasks for a heavy step: every row carries a history array many times its shuffle size, so merged
    partitions overrun executor memory (lost executors, then failed shuffle fetches)."""
    keys = ("spark.sql.shuffle.partitions", "spark.sql.adaptive.coalescePartitions.enabled")
    old = {k: spark.conf.get(k, None) for k in keys}
    spark.conf.set(keys[0], str(n)); spark.conf.set(keys[1], "false")
    try:
        yield
    finally:
        for k, v in old.items():
            spark.conf.unset(k) if v is None else spark.conf.set(k, v)

pr_built = not (exists(f"{P_PR}/_SUCCESS") and exists(f"{P_MO}/_SUCCESS"))
if pr_built:
    with timed("§2 predictions"), many_partitions(FC_PARTITIONS):
        E = F.expr
        sd = spark.read.parquet(P_SD).where(F.col("flow").isin("in", "out", "book"))
        cand = sd.groupBy(*SK).agg(F.count("*").alias("n"), F.min("txn_date").alias("first_dt"), F.max("txn_date").alias("last_dt"),
                                   F.max("payer_s").alias("payer_s"), F.max("payee_s").alias("payee_s")) \
                 .where(F.col("n") >= MIN_PAYMENTS)
        # streams silent since well before the judged months are stopped for every judged row (even a quarterly stream
        # is stopped after SKIP_MULT × 98 days), so they add nothing: left out
        cand = cand.where(F.col("last_dt") >= F.lit(E0 - dt.timedelta(days=int(SKIP_MULT * 98) + 1)))
        pay = (sd.join(cand.select(*SK), SK)
                 .select(*SK, "txn_date", F.lit("pay").alias("kind"), F.lit(1).alias("ko"), "amt", "rail"))
        mo = spark.createDataFrame([(m, month_first(m)) for m in EVAL], "month string, txn_date date")
        marks = (cand.select(*SK, "first_dt").crossJoin(F.broadcast(mo)).where(F.col("txn_date") > F.col("first_dt"))
                     .select(*SK, "txn_date", F.lit("month").alias("kind"), F.lit(0).alias("ko"),
                             F.lit(None).cast("double").alias("amt"), F.lit(None).cast("string").alias("rail")))
        ends = cand.select(*SK, F.lit(W_END + dt.timedelta(days=1)).alias("txn_date"), F.lit("end").alias("kind"),
                           F.lit(2).alias("ko"), F.lit(None).cast("double").alias("amt"), F.lit(None).cast("string").alias("rail"))
        R = (pay.unionByName(marks).unionByName(ends)
                .withColumn("d", F.datediff("txn_date", F.lit(EPOCH))).withColumn("b", bidx(F.col("txn_date"))))
        w = Window.partitionBy(*SK).orderBy("txn_date", "ko").rowsBetween(-H_ROWS, -1)
        R = R.withColumn("h", F.collect_list(F.when(F.col("kind") == "pay", F.struct("d", "b", "amt", "rail"))).over(w))
        # judged rows only: payments and month starts in the judged months, and the window end
        R = R.where((F.col("txn_date") >= F.lit(E0)) | (F.col("kind") == "end"))
        ws = F.datediff(F.add_months(F.trunc("txn_date", "month"), -REG_WINDOW), F.lit(EPOCH))
        R = (R.withColumn("ws", ws).withColumn("hf", E("filter(array_sort(h), x -> x.d >= ws)"))
              .withColumn("npay", F.size("hf"))
              .withColumn("hh", E(f"slice(hf, greatest(size(hf) - {REG_MAX_DAYS} + 1, 1), {REG_MAX_DAYS})"))
              .withColumn("dd", E("transform(hh, x -> x.d)")).withColumn("bb", E("transform(hh, x -> x.b)"))
              .withColumn("aa", E("transform(hh, x -> x.amt)"))
              .withColumn("g", E(_gaps("dd"))).withColumn("gb", E(_gaps("bb")))
              .withColumn("med", E(_nth("g", 0.5))).withColumn("medb", E(_nth("gb", 0.5)))
              .withColumn("daily", F.coalesce(F.col("medb") <= 1, F.lit(False)))
              .withColumn("tol", F.greatest(F.lit(float(TOL_MIN)), F.col("med") * float(TOL_FRAC)))
              .withColumn("ont", E("if(size(g) = 0, null, if(daily, size(filter(gb, x -> x <= 1)) / size(gb), "
                                   "size(filter(g, x -> abs(x - med) <= tol)) / size(g)))")))
        cad = F.lit(None).cast("string")
        for g_, lo, hi in CADENCES:
            cad = F.when(F.col("med").between(lo, hi), F.lit(g_)).otherwise(cad)
        cad = F.when(F.col("daily"), F.lit("daily")).otherwise(cad)
        R = (R.withColumn("cadence", cad)
              .withColumn("reg", F.coalesce((F.col("npay") >= MIN_PAYMENTS) & F.col("cadence").isNotNull()
                                            & (F.col("ont") >= REG_MIN), F.lit(False)))
              .where("reg")
              .withColumn("meda", E(_nth("aa", 0.5))).withColumn("p10a", E(_nth("aa", 0.1))).withColumn("p90a", E(_nth("aa", 0.9)))
              .withColumn("mean_a", E("aggregate(aa, cast(0 as double), (s, x) -> s + x) / size(aa)"))
              .withColumn("last_a", E("element_at(aa, -1)"))
              .withColumn("med3", E(_nth("slice(aa, greatest(size(aa) - 2, 1), 3)", 0.5)))
              .withColumn("mad", E(_nth("transform(aa, x -> abs(x - meda))", 0.5)))
              .withColumn("sigma", 1.4826 * F.col("mad"))                     # robust SD of one payment
              .withColumn("sd_a", E("sqrt(aggregate(aa, cast(0 as double), (s, x) -> s + (x - mean_a) * (x - mean_a)) / size(aa))"))
              .withColumn("p10g", E(_nth("g", 0.1))).withColumn("p90g", E(_nth("g", 0.9)))
              .withColumn("hn", F.size("aa"))
              .withColumn("lastd", E("element_at(dd, -1)")).withColumn("lastb", E("element_at(bb, -1)"))
              .withColumn("last_rail", E("element_at(hh, -1).rail"))
              .withColumn("low_thr", F.least(2 * F.col("p10a") - F.col("meda"), F.col("meda") * (1 - FIXED_PCT)))
              .drop("h", "hf", "hh", "dd", "bb", "aa", "g", "gb", "ws"))
        sk = float(SKIP_MULT)
        # payments: timing outcome against the prediction made after the previous payment
        gap, gapb = F.col("d") - F.col("lastd"), F.col("b") - F.col("lastb")
        timing = (F.when(F.col("daily"), F.when(gapb <= 1, "on time").otherwise("skipped"))
                   .when(gap >= sk * F.col("med"), "skipped")
                   .when(F.abs(gap - F.col("med")) <= F.col("tol"), "on time")
                   .when(gap < F.col("med"), "early").otherwise("late"))
        P = (R.where(F.col("kind") == "pay")
              .withColumn("outcome", timing).withColumn("gap", gap)
              .withColumn("date_err", F.when(F.col("daily"), gapb - 1).otherwise(gap - F.col("med")).cast("int"))
              .withColumn("pred_d", F.when(F.col("daily"), F.lit(None)).otherwise(F.col("lastd") + F.col("med")))
              .withColumn("low", ~F.col("daily") & (F.col("amt") < F.col("low_thr"))))
        # window end: a regular stream with no payment for SKIP_MULT × its gap (daily: DAILY_STOP_BD business days) has
        # stopped; counted once, as an outcome of the prediction after its last payment, if that prediction fell in
        # the judged months
        END_D, END_B = (W_END - EPOCH).days, bidx_py(W_END)
        stop = F.when(F.col("daily"), F.lit(END_B) - F.col("lastb") >= DAILY_STOP_BD) \
                .otherwise(F.lit(END_D) - F.col("lastd") >= sk * F.col("med"))
        exp_d = F.when(F.col("daily"), F.col("lastd") + 1).otherwise(F.col("lastd") + F.col("med"))
        X = (R.where((F.col("kind") == "end") & stop & (exp_d >= F.lit((E0 - EPOCH).days)))
              .withColumn("outcome", F.lit("stopped")).withColumn("date_err", F.lit(None).cast("int")).withColumn("gap", F.lit(None).cast("int"))
              .withColumn("pred_d", exp_d).withColumn("low", F.lit(False)).withColumn("amt", F.lit(None).cast("double"))
              .withColumn("txn_date", F.expr("date_add(to_date('1970-01-01'), cast(pred_d as int))")))
        keep = [*SK, "kind", "txn_date", "amt", "rail", "last_rail", "cadence", "daily", "npay", "med", "tol", "meda", "p10a",
                "p90a", "mean_a", "last_a", "med3", "mad", "sigma", "hn", "p10g", "p90g", "gap", "lastd", "low_thr", "outcome", "date_err", "pred_d", "low"]
        (P.select(*keep).unionByName(X.select(*keep))
          .join(cand.select(*SK, "payer_s", "payee_s"), SK)
          .withColumn("month", F.date_format("txn_date", "yyyy-MM"))
          .write.mode("overwrite").parquet(P_PR))
        # month starts: v4's expectation for the month (streams regular and not stopped at the end of last month)
        pend_d = F.col("d") - 1
        pend_b = bidx(F.date_sub(F.col("txn_date"), 1))
        stop_p = F.when(F.col("daily"), pend_b - F.col("lastb") >= DAILY_STOP_BD) \
                  .otherwise(pend_d - F.col("lastd") >= sk * F.col("med"))
        bd_map = F.create_map(*[F.lit(x) for kv in BD_IN.items() for x in kv])
        # projected payment dates: monthly + k months on the same day, bimonthly + 2k, quarterly + 3k, weekly / biweekly
        # + k × median gap; a date on a Saturday or Sunday moves to Monday
        proj = ("filter(transform(transform(sequence(1, 12), k -> case "
                "when cadence = 'monthly' then add_months(ld, k) "
                "when cadence = 'quarterly' and med < 70 then add_months(ld, 2 * k) "
                "when cadence = 'quarterly' then add_months(ld, 3 * k) "
                "else date_add(ld, cast(k * med as int)) end), "
                "x -> case dayofweek(x) when 7 then date_add(x, 2) when 1 then date_add(x, 1) else x end), "
                "x -> x >= txn_date and x <= last_day(txn_date))")
        M = (R.where((F.col("kind") == "month") & ~stop_p)
              .withColumn("month", F.date_format("txn_date", "yyyy-MM"))
              .withColumn("ld", F.expr("date_add(to_date('1970-01-01'), lastd)"))
              .withColumn("n_exp", F.when(F.col("daily"), bd_map[F.col("month")].cast("double"))
                                    .otherwise(F.dayofmonth(F.last_day("txn_date")) / F.col("med")))
              .withColumn("n_proj", F.when(F.col("daily"), bd_map[F.col("month")].cast("double")).otherwise(F.size(E(proj)).cast("double")))
              .withColumn("exp_v4", F.col("meda") * F.col("n_exp"))
              .withColumn("exp_dmed", F.col("meda") * F.col("n_proj"))
              .withColumn("exp_dmean", F.col("mean_a") * F.col("n_proj"))
              .withColumn("half90", 1.645 * F.col("sd_a") * F.sqrt("n_proj"))
              .withColumn("lo90", F.greatest(F.col("exp_dmean") - F.col("half90"), F.lit(0.0)))
              .withColumn("hi90", F.col("exp_dmean") + F.col("half90")))
        rec = (sd.withColumn("month", F.date_format("txn_date", "yyyy-MM")).where(F.col("month").isin(EVAL))
                 .groupBy(*SK, "month").agg(F.sum("amt").alias("rec_amt"), F.count("*").alias("n_rec")))
        # stopped for good: silent from the last payment in the window to the window end, long enough to call it a stop;
        # flagged in the month of that last payment and after
        END_D, END_B = (W_END - EPOCH).days, bidx_py(W_END)
        last_d, last_b = F.datediff("last_dt", F.lit(EPOCH)), bidx(F.col("last_dt"))
        confirmed = F.when(F.col("daily"), F.lit(END_B) - last_b >= DAILY_STOP_BD).otherwise(F.lit(END_D) - last_d >= sk * F.col("med"))
        (M.select(*SK, "month", "cadence", "daily", "med", "meda", "mean_a", "mad", "sd_a", F.col("last_rail").alias("rail"),
                  "n_exp", "n_proj", "exp_v4", "exp_dmed", "exp_dmean", "lo90", "hi90")
          .join(rec, SK + ["month"], "left").fillna({"rec_amt": 0.0, "n_rec": 0})
          .join(cand.select(*SK, "payer_s", "payee_s", "last_dt"), SK)
          .withColumn("stop_good", F.coalesce(confirmed & (F.col("month") >= F.date_format("last_dt", "yyyy-MM")), F.lit(False)))
          .write.mode("overwrite").parquet(P_MO))

PR = spark.read.parquet(P_PR)
MO = spark.read.parquet(P_MO)
def persp(df):
    """One row per customer side: in ext, out ext, and book streams once from each sampled side (in int, out int)."""
    sides = F.filter(F.array(
        F.when((F.col("flow") == "in") & F.col("payee_s"), F.struct(F.lit("in ext").alias("side"), F.col("payee").alias("cust"))),
        F.when((F.col("flow") == "out") & F.col("payer_s"), F.struct(F.lit("out ext").alias("side"), F.col("payer").alias("cust"))),
        F.when((F.col("flow") == "book") & F.col("payee_s"), F.struct(F.lit("in int").alias("side"), F.col("payee").alias("cust"))),
        F.when((F.col("flow") == "book") & F.col("payer_s"), F.struct(F.lit("out int").alias("side"), F.col("payer").alias("cust")))),
        lambda x: x.isNotNull())
    return df.withColumn("_s", F.explode(sides)).withColumn("side", F.col("_s.side")).withColumn("cust", F.col("_s.cust")).drop("_s")
PRS = persp(PR).withColumn("rail", F.coalesce("rail", "last_rail")).cache()
MOS = persp(MO).cache()
SIDES = ["in ext", "out ext", "in int", "out int"]
s2 = (PRS.groupBy("side").agg(F.sum((F.col("kind") == "pay").cast("long")).alias("payments judged"),
                              F.sum((F.col("outcome") == "stopped").cast("long")).alias("stops judged"),
                              F.countDistinct(*SK).alias("regular streams"))
         .join(MOS.groupBy("side").agg(F.count("*").alias("stream-months judged")), "side", "outer").toPandas())
s2["side"] = pd.Categorical(s2.side, SIDES); s2 = s2.sort_values("side")
out(sec(f"§2 Predictions, judged {EVAL[0]} → {EVAL[-1]} — " + (f"built ({TIMES['§2 predictions']:.1f} min)" if pr_built else "reused"),
        tbl(s2)), para(f"Scratch {P_PR} and {P_MO}. A book stream is counted from each sampled side."))

## §3 Timing

Outcome of each prediction "the next payment comes about one usual gap after the last one": on time, early, late,
skipped (gap ≥ `SKIP_MULT ×` median), or stopped (no payment by the end of the window). Days off = actual gap −
median gap (daily streams: business days after the next business day). Shares are of predictions. **Date interval:**
share of arrived payments whose gap fell between the p10 and p90 of the past gaps, against the share it would hold if
nothing changed (non-daily).

In [ ]:
# ---- §3 Timing ----------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

OUTCOMES = ["on time", "early", "late", "skipped", "stopped"]
ERR_BINS = [("≤ −4", None, -4), ("−3..−1", -3, -1), ("0", 0, 0), ("+1..+2", 1, 2), ("+3..+5", 3, 5), ("+6..+10", 6, 10),
            ("> +10", 11, None)]
def order(df, col, cats):
    df[col] = pd.Categorical(df[col].astype(str), cats + sorted(set(df[col].astype(str)) - set(cats)))
    return df.sort_values(col)
def timing(keys):
    ad = F.abs(F.col("date_err"))
    t = (PRS.groupBy(*keys).agg(F.count("*").alias("predictions"),
                                *[F.avg((F.col("outcome") == o).cast("double")).alias(o) for o in OUTCOMES],
                                F.percentile_approx(ad, 0.5, 1000).alias("median days off"),
                                F.percentile_approx(ad, 0.9, 1000).alias("p90 days off"),
                                F.avg(F.when(~F.col("daily") & F.col("gap").isNotNull(),
                                             F.col("gap").between(F.col("p10g"), F.col("p90g")).cast("double"))).alias("gap in p10–p90 of past gaps"),
                                F.avg(F.when(~F.col("daily") & F.col("gap").isNotNull(),
                                             (F.floor((F.col("hn") - 2) * 0.9) - F.floor((F.col("hn") - 2) * 0.1)) / F.col("hn"))).alias("if nothing changed"))
            .toPandas())
    for k in keys:
        t = order(t, k, {"cadence": CAD_GROUPS, "side": SIDES}.get(k, []))
    return t
t3a = timing(["cadence"]); t3b = timing(["side", "cadence"]); t3c = timing(["rail"]).sort_values("predictions", ascending=False)
t3d = (PRS.where(F.col("kind") == "pay").groupBy("cadence", "rail").agg(F.avg((F.col("outcome") == "on time").cast("double")).alias("on time"),
                                                                          F.count("*").alias("n")).toPandas())
t3d = t3d[t3d.n >= 100].pivot(index="cadence", columns="rail", values="on time").reindex([c for c in CAD_GROUPS if c in set(t3d.cadence)]).reset_index()
def err_cond(lo, hi):
    c = F.col("date_err").isNotNull()
    if lo is not None: c = c & (F.col("date_err") >= lo)
    if hi is not None: c = c & (F.col("date_err") <= hi)
    return c
t3e = order(PRS.where(F.col("date_err").isNotNull()).groupBy("cadence")
               .agg(*[F.avg(err_cond(lo, hi).cast("double")).alias(l) for l, lo, hi in ERR_BINS]).toPandas(), "cadence", CAD_GROUPS)
t3f = (PRS.groupBy("month").agg(F.count("*").alias("predictions"), *[F.avg((F.col("outcome") == o).cast("double")).alias(o) for o in OUTCOMES])
          .toPandas().sort_values("month"))
PCT_O = tuple(OUTCOMES) + ("gap in p10–p90 of past gaps", "if nothing changed")
out(sec("§3a Timing outcome by cadence", tbl(t3a, pct=PCT_O)),
    sec("§3b By side and cadence", tbl(t3b, pct=PCT_O)),
    sec("§3c By rail (rail of the payment; stops: rail of the last payment)", tbl(t3c, pct=PCT_O)),
    sec("§3d On-time share by cadence and rail (cells with ≥ 100 payments)", tbl(t3d, pct=tuple(t3d.columns[1:]))),
    sec("§3e Days off the usual gap, payments that arrived (daily: business days)", tbl(t3e, pct=tuple(l for l, _, _ in ERR_BINS))),
    sec("§3f Timing outcome by month", tbl(t3f, pct=PCT_O)))

## §4 Amount

For every payment that arrived: error = (actual − predicted) / predicted. **Median** is v4's predictor; the others are
for comparison. *Within ±x* = share of payments within x of the prediction; *dollar error* = sum of |actual −
median| / sum of actual (large payments weigh more). *Band* = share of payments inside the history's p10–p90 range
(with a full 26-payment history about 74% when nothing changes, because the band runs from the 3rd to the 23rd
value; *band if nothing changed* gives that share for the actual history lengths, so a gap between the two means the
history misses something); *band width* = (p90 − p10) / median. A second interval, *median ± 1.645σ* with σ = 1.4826 ×
MAD (a robust standard deviation), should hold about 90%; fixed-amount streams have σ = 0, so any change falls outside.
*Flagged low* = v4's amount-low rule (non-daily). **Spread**
MAD / median of the history groups streams by how variable they were before the payment.

In [ ]:
# ---- §4 Amount ----------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

PREDS = {"median (v4)": "meda", "last payment": "last_a", "mean": "mean_a", "median of last 3": "med3"}
WITHIN = [("±1%", 0.01), ("±5%", 0.05), ("±10%", 0.10), ("±25%", 0.25)]
SPREAD = [("0", 0, 0), ("≤ 0.5%", 0, 0.005), ("0.5–2%", 0.005, 0.02), ("2–5%", 0.02, 0.05), ("5–10%", 0.05, 0.10),
          ("10–25%", 0.10, 0.25), ("> 25%", 0.25, None)]
A = (PRS.where((F.col("kind") == "pay") & (F.col("meda") > 0))
        .withColumn("err", (F.col("amt") - F.col("meda")) / F.col("meda")).withColumn("ape", F.abs(F.col("err")))
        .withColumn("rmad", F.col("mad") / F.col("meda"))
        # share of new values a p10–p90 band of hn past values holds when nothing changes (lower-element quantiles, as v4)
        .withColumn("band_exp", (F.floor((F.col("hn") - 1) * 0.9) - F.floor((F.col("hn") - 1) * 0.1)) / (F.col("hn") + 1)))
sp = F.lit(None).cast("string")
for l, lo, hi in SPREAD:
    c = (F.col("rmad") == 0) if hi == 0 else ((F.col("rmad") > lo) & (F.col("rmad") <= hi) if hi is not None else F.col("rmad") > lo)
    sp = F.when(c, F.lit(l)).otherwise(sp)
A = A.withColumn("spread", sp).cache()
def amount(keys):
    t = (A.groupBy(*keys).agg(F.count("*").alias("payments"),
                              F.percentile_approx("ape", 0.5, 1000).alias("median error"),
                              F.percentile_approx("err", 0.5, 1000).alias("median signed error"),
                              *[F.avg((F.col("ape") <= x).cast("double")).alias(f"within {l}") for l, x in WITHIN],
                              (F.sum(F.abs(F.col("amt") - F.col("meda"))) / F.sum("amt")).alias("dollar error"),
                              F.avg(((F.col("amt") >= F.col("p10a")) & (F.col("amt") <= F.col("p90a"))).cast("double")).alias("in p10–p90 band"),
                              F.avg(F.col("band_exp")).alias("band if nothing changed"),
                              F.percentile_approx((F.col("p90a") - F.col("p10a")) / F.col("meda"), 0.5, 1000).alias("band width"),
                              F.avg((F.col("ape") * F.col("meda") <= 1.645 * F.col("sigma")).cast("double")).alias("in median ± 1.645σ (90%)"),
                              F.percentile_approx(2 * 1.645 * F.col("sigma") / F.col("meda"), 0.5, 1000).alias("± 1.645σ width"),
                              F.avg(F.when(~F.col("daily"), F.col("low").cast("double"))).alias("flagged low"))
            .toPandas())
    for k in keys:
        t = order(t, k, {"cadence": CAD_GROUPS, "side": SIDES, "spread": [l for l, _, _ in SPREAD]}.get(k, []))
    return t
PCT_A = ("median error", "median signed error", *[f"within {l}" for l, _ in WITHIN], "dollar error", "in p10–p90 band", "band if nothing changed", "band width", "in median ± 1.645σ (90%)", "± 1.645σ width", "flagged low")
t4a = amount(["cadence"]); t4b = amount(["side", "cadence"]); t4c = amount(["rail"]).sort_values("payments", ascending=False)
t4d = amount(["spread"]); t4e = amount(["cadence", "spread"])
t4e = t4e[t4e.payments >= 100]
pc = []
for name, c in PREDS.items():
    e_ = F.abs(F.col("amt") - F.col(c)) / F.col(c)
    r = (A.where(F.col(c) > 0).groupBy("cadence").agg(F.percentile_approx(e_, 0.5, 1000).alias("median error"),
                                                     F.avg((e_ <= 0.05).cast("double")).alias("within ±5%"),
                                                     (F.sum(F.abs(F.col("amt") - F.col(c))) / F.sum("amt")).alias("dollar error"))
          .toPandas())
    r["predictor"] = name; pc.append(r)
t4f = order(pd.concat(pc), "cadence", CAD_GROUPS)
t4f = t4f.pivot(index="cadence", columns="predictor", values=["median error", "within ±5%", "dollar error"])
t4f.columns = [f"{b}: {a}" for a, b in t4f.columns]
t4f = t4f[[f"{p}: {m}" for m in ["median error", "within ±5%", "dollar error"] for p in PREDS]].reset_index()
out(sec("§4a Next amount against the median, by cadence", tbl(t4a, pct=PCT_A)),
    sec("§4b By side and cadence", tbl(t4b, pct=PCT_A)),
    sec("§4c By rail", tbl(t4c, pct=PCT_A)),
    sec("§4d By spread of the history (MAD / median): does the spread tell how far to trust the median?", tbl(t4d, pct=PCT_A)),
    sec("§4e By cadence and spread (cells with ≥ 100 payments)", tbl(t4e, pct=PCT_A)),
    sec("§4f Predictors compared, by cadence", tbl(t4f, pct=tuple(t4f.columns[1:]))))

## §5 Expected dollars per month

Three ways to set a stream's expected dollars for a month (streams regular and not stopped at the end of the previous
month):

| method | payments expected in the month | amount per payment |
|---|---|---|
| `v4` (what customer metrics v4 stores) | days in month / median gap (daily: business days) | median |
| `dates × median` | payment dates projected from the last payment: monthly + 1, 2, … months on the same day, quarterly + 3 months (bimonthly + 2), weekly / biweekly + the median gap; a date on a weekend moves to Monday; daily: business days | median |
| `dates × mean` | the same projected dates | mean (expected totals of variable amounts) |

**Stops for good.** A stream whose last payment in the window is followed by silence until the window end (at least
`SKIP_MULT ×` its gap; daily: `DAILY_STOP_BD` business days) stopped for good. In the month of its last payment and
after, the expected dollars that did not come are a real event, not a forecasting error: that shortfall is shown
separately and taken out in the *excl. stops* columns (dollar measures), and those stream-months are left out of the
*within* shares. Customer level takes the shortfall out of the customer's expected dollars.

**Measures.** *Received / expected* = total received / total expected. *Dollar error* = sum of |received − expected|
/ sum of received; *excl. stops* takes out the stops' shortfall; *from stops* = the stops' share of the error.
*Within ±10%* = |received − expected| ≤ 10% of expected (expected 0: nothing came). *90% interval* (dates × mean):
expected ± 1.645 × SD of the history's amounts × √(payments expected); *coverage* = share of stream-months whose
received dollars fall inside (90% if it holds); *width* = interval width / expected (median).

In [ ]:
# ---- §5 Expected dollars per month --------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

METHODS = {"v4": "exp_v4", "dates × median": "exp_dmed", "dates × mean": "exp_dmean"}
def add_stop(df):
    for c in METHODS.values():
        df = df.withColumn(f"ss_{c}", F.when(F.col("stop_good") & (F.col(c) > F.col("rec_amt")), F.col(c) - F.col("rec_amt")).otherwise(0.0))
    return df
def mstats(df, c, pre=""):
    e, ss = F.col(c), F.col(f"ss_{c}")
    err = F.abs(F.col("rec_amt") - e)
    hit = F.when(e > 0, err <= 0.10 * e).otherwise(F.col("rec_amt") == 0)
    return [(F.sum("rec_amt") / F.sum(e)).alias(f"{pre}received / expected"),
            (F.sum(err) / F.sum("rec_amt")).alias(f"{pre}dollar error"),
            ((F.sum(err) - F.sum(ss)) / F.sum("rec_amt")).alias(f"{pre}dollar error excl. stops"),
            (F.sum(ss) / F.sum(err)).alias(f"{pre}from stops"),
            F.avg(F.when(~F.col("stop_good"), hit.cast("double"))).alias(f"{pre}within ±10%")]
def order_keys(t, keys):
    for k in keys:
        t = order(t, k, {"cadence": CAD_GROUPS, "side": SIDES}.get(k, []))
    return t
def monthly(df, keys, unit):
    t = (df.groupBy(*keys).agg(F.count("*").alias(unit), (F.sum("exp_v4") / 1e6).alias("expected (USD mn)"),
                               (F.sum("rec_amt") / 1e6).alias("received (USD mn)"), *mstats(df, "exp_v4"),
                               F.avg((F.col("rec_amt") == 0).cast("double")).alias("nothing came"),
                               F.avg(F.col("stop_good").cast("double")).alias("stopped for good"))
           .toPandas())
    return order_keys(t, keys)
MOS2 = add_stop(MOS).cache()
PCT_M = ("dollar error", "dollar error excl. stops", "from stops", "within ±10%", "nothing came", "stopped for good")
DEC_M = ("expected (USD mn)", "received (USD mn)", "received / expected")
t5a = monthly(MOS2, ["cadence"], "stream-months"); t5b = monthly(MOS2, ["side", "cadence"], "stream-months")
t5c = monthly(MOS2, ["rail"], "stream-months").sort_values("stream-months", ascending=False)
t5d = monthly(MOS2, ["month", "cadence"], "stream-months").sort_values(["month", "cadence"])
CM = (MOS2.groupBy("cust", "side", "month")
          .agg(*[F.sum(c).alias(c) for c in METHODS.values()], *[F.sum(f"ss_{c}").alias(f"ss_{c}") for c in METHODS.values()],
               F.sum("rec_amt").alias("rec_amt"))
          .withColumn("stop_good", F.lit(False)))
# customer level: the stops' shortfall is taken out of expected for the within share (other streams still count)
CMa = CM
for c in METHODS.values():
    CMa = CMa.withColumn(c, F.col(c) - F.col(f"ss_{c}")).withColumn(f"ss_{c}", F.lit(0.0))
def monthly_c(keys):
    a = CM.groupBy(*keys).agg(F.count("*").alias("customer-months"), (F.sum("exp_v4") / 1e6).alias("expected (USD mn)"),
                              (F.sum("rec_amt") / 1e6).alias("received (USD mn)"), *mstats(CM, "exp_v4")[:4],
                              F.avg((F.col("rec_amt") == 0).cast("double")).alias("nothing came")).toPandas()
    b = CMa.groupBy(*keys).agg(*mstats(CMa, "exp_v4")[4:]).toPandas()
    return order_keys(a.merge(b, on=keys), keys)
t5e = monthly_c(["side"]); t5f = monthly_c(["month", "side"]).sort_values(["month", "side"])
def methods(df, keys, unit, adj=None):
    t = df.groupBy(*keys).agg(F.count("*").alias(unit), *[x for nm_, c in METHODS.items() for x in mstats(df, c, f"{nm_}: ")[:4]]).toPandas()
    if adj is not None:
        b = adj.groupBy(*keys).agg(*[mstats(adj, c, f"{nm_}: ")[4] for nm_, c in METHODS.items()]).toPandas()
    else:
        b = df.groupBy(*keys).agg(*[mstats(df, c, f"{nm_}: ")[4] for nm_, c in METHODS.items()]).toPandas()
    t = order_keys(t.merge(b, on=keys), keys)
    cols = [f"{nm_}: {m}" for m in ["received / expected", "dollar error", "dollar error excl. stops", "within ±10%"] for nm_ in METHODS]
    return t[keys + [unit] + cols]
t5g = methods(MOS2, ["cadence"], "stream-months"); t5h = methods(CM, ["side"], "customer-months", CMa)
t5i = methods(MOS2, ["side", "cadence"], "stream-months")
iv = MOS2.where(~F.col("stop_good") & (F.col("exp_dmean") > 0))
inside = ((F.col("rec_amt") >= F.col("lo90")) & (F.col("rec_amt") <= F.col("hi90"))).cast("double")
t5j = order_keys(iv.groupBy("cadence").agg(F.count("*").alias("stream-months"), F.avg(inside).alias("coverage (90% nominal)"),
                                         F.avg(F.when(F.col("rec_amt") < F.col("lo90"), 1.0).otherwise(0.0)).alias("below"),
                                         F.avg(F.when(F.col("rec_amt") > F.col("hi90"), 1.0).otherwise(0.0)).alias("above"),
                                         F.percentile_approx((F.col("hi90") - F.col("lo90")) / F.col("exp_dmean"), 0.5, 1000).alias("width"))
                          .toPandas(), ["cadence"])
PCT_G = tuple(c for c in t5g.columns if not c.endswith("received / expected") and c not in ("cadence", "side", "stream-months", "customer-months"))
DEC_G = tuple(c for c in t5g.columns if c.endswith("received / expected"))
out(sec("§5a v4 expectation, stream level, by cadence", tbl(t5a, pct=PCT_M, dec=DEC_M)),
    sec("§5b v4, by side and cadence", tbl(t5b, pct=PCT_M, dec=DEC_M)),
    sec("§5c v4, by rail of the last payment", tbl(t5c, pct=PCT_M, dec=DEC_M)),
    sec("§5d v4, by month and cadence", tbl(t5d, pct=PCT_M, dec=DEC_M)),
    sec("§5e v4, customer level (a side's regular streams added), by side", tbl(t5e, pct=PCT_M, dec=DEC_M)),
    sec("§5f v4, customer level by month and side", tbl(t5f, pct=PCT_M, dec=DEC_M)),
    sec("§5g Methods compared, stream level, by cadence", tbl(t5g, pct=PCT_G, dec=DEC_G)),
    sec("§5h Methods compared, customer level, by side", tbl(t5h, pct=PCT_G, dec=DEC_G)),
    sec("§5i Methods compared, by side and cadence", tbl(t5i, pct=PCT_G, dec=DEC_G)),
    sec("§5j 90% interval of the month's dollars (dates × mean), stops for good left out", tbl(t5j, pct=("coverage (90% nominal)", "below", "above", "width"))))

## §6 Figures

Saves the main results as PNG files in `FIG_DIR/<window end>/` next to the notebook (nothing drawn inline), including
example streams over time: each payment (dot) against the predicted amount (the median of the history before it,
line) and its p10–p90 band; late, skipped and amount-low payments are marked. Examples are picked by a hash of the
stream, so a rerun draws the same ones.

In [ ]:
# ---- §6 Figures ---------------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"

import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter

FDIR = os.path.join(FIG_DIR, END)
os.makedirs(FDIR, exist_ok=True)
SURF, INK, INK2, GRID, NEUTRAL = "#fcfcfb", "#0b0b0b", "#52514e", "#e3e2de", "#c3c2b7"
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]   # fixed order
plt.rcParams.update({"figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
                     "axes.edgecolor": NEUTRAL, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True,
                     "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold", "axes.titlelocation": "left",
                     "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
                     "legend.frameon": False, "lines.linewidth": 2})
CAD_COLOR = dict(zip(CAD_GROUPS, [PAL[0], PAL[1], PAL[2], PAL[3], PAL[6]]))   # a cadence keeps its colour everywhere
OUT_COLOR = dict(zip(OUTCOMES, [PAL[2], PAL[0], PAL[3], PAL[1], PAL[7]]))
SIDE_COLOR = dict(zip(SIDES, [PAL[0], PAL[1], PAL[2], PAL[4]]))
PRED_COLOR = dict(zip(PREDS, [PAL[0], PAL[1], PAL[2], PAL[6]]))
SAVED = []
def save(fig, name, what):
    fig.savefig(os.path.join(FDIR, name), dpi=160, bbox_inches="tight")
    plt.close(fig)
    SAVED.append({"file": name, "shows": what})
def stacked(ax, labels, parts, names, colors, min_label=0.08):
    labels = [str(x) for x in labels]
    y = np.arange(len(labels)); left = np.zeros(len(labels))
    for vals, nm_, col in zip(parts, names, colors):
        v = np.nan_to_num(np.asarray(vals, dtype=float))
        ax.barh(y, v, left=left, color=col, edgecolor=SURF, linewidth=2, height=0.62, label=nm_)
        for i, (l0, w_) in enumerate(zip(left, v)):
            if w_ >= min_label:
                ax.text(l0 + w_ / 2, i, f"{w_:.0%}", ha="center", va="center", fontsize=7.5, color=INK)
        left += v
    ax.set_yticks(y); ax.set_yticklabels(labels); ax.invert_yaxis()
    ax.set_xlim(0, 1); ax.xaxis.set_major_formatter(PercentFormatter(1.0)); ax.grid(axis="y", visible=False)
def grouped(ax, cats, series, colors, fmt_pct=True, w=None):
    n = len(series); w = w or 0.8 / n
    for i, (nm_, vals) in enumerate(series.items()):
        ax.bar(np.arange(len(cats)) + (i - (n - 1) / 2) * w, vals, width=w, color=colors[nm_], edgecolor=SURF, linewidth=1, label=nm_)
    ax.set_xticks(range(len(cats))); ax.set_xticklabels(cats); ax.grid(axis="x", visible=False)
    if fmt_pct:
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
tag = f"judged {EVAL[0]} to {EVAL[-1]}, {SAMPLE_PCT}% of customers"
cads = [str(c) for c in t3a.cadence]

# 1 timing outcome by cadence and by side
fig, axs = plt.subplots(1, 2, figsize=(11, 2.9))
stacked(axs[0], cads, [t3a[o] for o in OUTCOMES], OUTCOMES, [OUT_COLOR[o] for o in OUTCOMES]); axs[0].set_title("By cadence")
ts = timing(["side"])
stacked(axs[1], [str(s) for s in ts.side], [ts[o] for o in OUTCOMES], OUTCOMES, [OUT_COLOR[o] for o in OUTCOMES]); axs[1].set_title("By side")
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.04))
fig.suptitle(f"Did the next payment come when expected? ({tag})", x=0.01, ha="left", y=1.04)
fig.subplots_adjust(bottom=0.22, wspace=0.35); save(fig, "01_timing_outcome.png", "on time / early / late / skipped / stopped by cadence and side")

# 2 days off the usual gap
labs = [l for l, _, _ in ERR_BINS]; rows_ = list(t3e.iterrows())
fig, axs = plt.subplots(1, len(rows_), figsize=(2.2 * len(rows_), 2.8), sharey=True)
axs = np.atleast_1d(axs)
for ax, (_, r) in zip(axs, rows_):
    ax.bar(range(len(labs)), [r[l] for l in labs], color=CAD_COLOR.get(str(r.cadence), NEUTRAL), width=0.75)
    ax.set_xticks(range(len(labs))); ax.set_xticklabels(labs, rotation=60, fontsize=7); ax.set_title(str(r.cadence)); ax.grid(axis="x", visible=False)
axs[0].yaxis.set_major_formatter(PercentFormatter(1.0)); axs[0].set_ylabel("share of payments")
fig.suptitle("Days off the usual gap (negative = early; daily: business days)", x=0.01, ha="left", y=1.05)
save(fig, "02_days_off.png", "distribution of days early / late by cadence")

# 3 amount: share within ±x of the median
xs = [l for l, _ in WITHIN]
fig, ax = plt.subplots(figsize=(7, 3.4))
for _, r in t4a.iterrows():
    ax.plot(range(len(xs)), [r[f"within {l}"] for l in xs], color=CAD_COLOR.get(str(r.cadence), NEUTRAL), marker="o", markersize=4, label=str(r.cadence))
ax.set_xticks(range(len(xs))); ax.set_xticklabels(xs); ax.set_ylim(0, 1.02); ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_xlabel("next payment within this much of the median of the history"); ax.set_ylabel("share of payments")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1.0)); ax.set_title("How close the next amount is to the median, by cadence")
save(fig, "03_amount_within.png", "share of next payments within ±x of the median, by cadence")

# 4 predictors compared
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
cc = [str(c) for c in t4f.cadence]
grouped(axs[0], cc, {p: t4f[f"{p}: median error"].values for p in PREDS}, PRED_COLOR); axs[0].set_title("Median error of the next amount (lower is better)")
grouped(axs[1], cc, {p: t4f[f"{p}: within ±5%"].values for p in PREDS}, PRED_COLOR); axs[1].set_title("Next amount within ±5% (higher is better)")
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=4, bbox_to_anchor=(0.5, -0.04))
fig.subplots_adjust(bottom=0.2, wspace=0.25); save(fig, "04_predictors.png", "median vs last payment vs mean vs median of last 3")

# 5 spread of the history vs error and band coverage
fig, axs = plt.subplots(1, 2, figsize=(11, 3.2))
spl = [str(s) for s in t4d.spread]
axs[0].plot(range(len(spl)), t4d["median error"], color=PAL[0], marker="o", markersize=4, label="median error")
axs[0].plot(range(len(spl)), 1 - t4d["within ±5%"], color=PAL[1], marker="o", markersize=4, label="share off by more than 5%")
axs[0].set_xticks(range(len(spl))); axs[0].set_xticklabels(spl); axs[0].yaxis.set_major_formatter(PercentFormatter(1.0))
axs[0].set_xlabel("spread of the history (MAD / median)"); axs[0].legend(loc="upper left"); axs[0].set_title("Error grows with the spread of the history")
axs[1].bar(range(len(spl)), t4d["in p10–p90 band"], color=PAL[2], width=0.7)
axs[1].scatter(range(len(spl)), t4d["band if nothing changed"], marker="_", s=600, color=INK, linewidths=2, zorder=3, label="if nothing changed")
axs[1].legend(loc="lower right")
axs[1].set_xticks(range(len(spl))); axs[1].set_xticklabels(spl); axs[1].set_ylim(0, 1); axs[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axs[1].set_xlabel("spread of the history (MAD / median)"); axs[1].set_title("Next payments inside the p10–p90 band"); axs[1].grid(axis="x", visible=False)
fig.subplots_adjust(wspace=0.25); save(fig, "05_spread_vs_error.png", "error and band coverage by the spread of the history")

# 6 monthly received / expected over time
ms = sorted(set(t5d.month.astype(str)))
fig, axs = plt.subplots(2, 1, figsize=(9, 5.4), sharex=True)
for c in CAD_GROUPS:
    x = t5d[t5d.cadence.astype(str) == c].set_index("month")
    if len(x):
        axs[0].plot(range(len(ms)), [x["received / expected"].get(m, np.nan) for m in ms], color=CAD_COLOR[c], marker="o", markersize=3, label=c)
        axs[1].plot(range(len(ms)), [x["dollar error"].get(m, np.nan) for m in ms], color=CAD_COLOR[c], marker="o", markersize=3, label=c)
axs[0].axhline(1, color=INK2, linestyle="--", linewidth=1); axs[0].set_title("Received / expected dollars (stream level)")
axs[1].yaxis.set_major_formatter(PercentFormatter(1.0)); axs[1].set_title("Dollar error |received − expected| / expected")
axs[1].set_xticks(range(len(ms))); axs[1].set_xticklabels(ms, rotation=45); axs[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
save(fig, "06_monthly_over_time.png", "monthly expected vs received over the judged months, by cadence")

# 7 customer level over time by side
fig, axs = plt.subplots(2, 1, figsize=(9, 5.4), sharex=True)
for s in SIDES:
    x = t5f[t5f.side.astype(str) == s].set_index("month")
    if len(x):
        axs[0].plot(range(len(ms)), [x["received / expected"].get(m, np.nan) for m in ms], color=SIDE_COLOR[s], marker="o", markersize=3, label=s)
        axs[1].plot(range(len(ms)), [x["within ±10%"].get(m, np.nan) for m in ms], color=SIDE_COLOR[s], marker="o", markersize=3, label=s)
axs[0].axhline(1, color=INK2, linestyle="--", linewidth=1); axs[0].set_title("Customer level: received / expected regular dollars")
axs[1].yaxis.set_major_formatter(PercentFormatter(1.0)); axs[1].set_ylim(0, 1); axs[1].set_title("Customer-months within ±10% of expected")
axs[1].set_xticks(range(len(ms))); axs[1].set_xticklabels(ms, rotation=45); axs[0].legend(loc="upper left", bbox_to_anchor=(1.01, 1.0))
save(fig, "07_customer_monthly.png", "customer-level expected vs received by side over time")

# 8 heatmap: on time and within ±5% by cadence and rail
hm_t = t3d.set_index("cadence")
am = (A.groupBy("cadence", "rail").agg(F.avg((F.col("ape") <= 0.05).cast("double")).alias("v"), F.count("*").alias("n")).toPandas())
hm_a = am[am.n >= 100].pivot(index="cadence", columns="rail", values="v").reindex(hm_t.index)
fig, axs = plt.subplots(1, 2, figsize=(11, 0.5 * len(hm_t) + 1.6))
for ax, hm_, title in [(axs[0], hm_t, "Next payment on time"), (axs[1], hm_a, "Next amount within ±5% of the median")]:
    v = hm_.values.astype(float)
    ax.imshow(np.nan_to_num(v, nan=0), cmap="Blues", vmin=0, vmax=1, aspect="auto")
    for i in range(v.shape[0]):
        for j in range(v.shape[1]):
            if not np.isnan(v[i, j]):
                ax.text(j, i, f"{v[i, j]:.0%}", ha="center", va="center", fontsize=8, color="white" if v[i, j] > 0.6 else INK)
    ax.set_xticks(range(v.shape[1])); ax.set_xticklabels(list(hm_.columns)); ax.set_yticks(range(v.shape[0])); ax.set_yticklabels(list(hm_.index))
    ax.grid(False); ax.set_title(title)
save(fig, "08_cadence_by_rail.png", "on-time and amount accuracy by cadence and rail (cells with ≥ 100 payments)")

# 9 example streams over time
ex_keys = (PR.where(F.col("kind") == "pay").groupBy(*SK, "cadence").agg(F.count("*").alias("n"),
              F.sum((F.col("outcome") != "on time").cast("int")).alias("miss"),
              F.percentile_approx(F.col("mad") / F.col("meda"), 0.5, 100).alias("rm"))
              .where(F.col("n") >= F.when(F.col("cadence") == "quarterly", 3).otherwise(6)).withColumn("h", F.expr("xxhash64(concat_ws('|', payer, payee, flow))"))
              .withColumn("kind_", F.when(F.col("miss") > 0, "timing miss").when(F.col("rm") > 0.02, "amount varies").otherwise("steady"))
              .withColumn("r", F.row_number().over(Window.partitionBy("cadence", "kind_").orderBy("h")))
              .where("r = 1").toPandas())
ex_keys["cadence"] = pd.Categorical(ex_keys.cadence, CAD_GROUPS)
ex_keys = ex_keys.sort_values(["cadence", "kind_"]).groupby("cadence", observed=True).head(EXAMPLES_PER_CAD)
n_ex = 0
for cad_, grp in ex_keys.groupby("cadence", observed=True):
    if not len(grp):
        continue
    sel = F.lit(False)
    for _, k in grp.iterrows():
        sel = sel | ((F.col("payer") == k.payer) & (F.col("payee") == k.payee) & (F.col("flow") == k.flow))
    pts = PR.where(sel & (F.col("kind") == "pay")).select(*SK, "txn_date", "amt", "meda", "p10a", "p90a", "outcome", "low").toPandas()
    fig, axs = plt.subplots(len(grp), 1, figsize=(9, 2.4 * len(grp)), squeeze=False)
    for ax, (_, k) in zip(axs[:, 0], grp.iterrows()):
        p = pts[(pts.payer == k.payer) & (pts.payee == k.payee) & (pts.flow == k.flow)].sort_values("txn_date")
        x = pd.to_datetime(p.txn_date)
        ax.fill_between(x, p.p10a, p.p90a, step="post", color=CAD_COLOR[str(cad_)], alpha=0.15, linewidth=0, label="p10–p90 of history")
        ax.step(x, p.meda, where="post", color=CAD_COLOR[str(cad_)], linewidth=1.5, label="predicted (median of history)")
        ok = p.outcome == "on time"
        ax.scatter(x[ok], p.amt[ok], s=12, color=INK, zorder=3, label="payment on time")
        ax.scatter(x[~ok], p.amt[~ok], s=28, color=PAL[1], marker="D", zorder=4, label="early / late / skipped")
        if p.low.any():
            ax.scatter(x[p.low], p.amt[p.low], s=60, facecolors="none", edgecolors=PAL[7], linewidths=1.5, zorder=5, label="flagged low (v4)")
        ax.set_title(f"{k.kind_}: {k.flow}, {k.payer} → {k.payee}", fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m")); ax.set_ylabel("USD")
    h, l = axs[0, 0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.02), fontsize=8)
    fig.suptitle(f"Example {cad_} streams: payments against the prediction", x=0.01, ha="left", y=1.0)
    fig.tight_layout(rect=(0, 0.05, 1, 0.98))
    n_ex += 1
    save(fig, f"09_examples_{str(cad_)}.png", f"example {cad_} streams over time ({len(grp)})")

# 10 monthly expectation methods: error split into forecast error and stops for good
MCOL = {"v4": PAL[7], "dates × median": PAL[0], "dates × mean": PAL[2]}
fig, axs = plt.subplots(1, 2, figsize=(12, 3.4), gridspec_kw={"width_ratios": [len(t5g), max(len(t5h), 2)]})
for ax, t, key in [(axs[0], t5g, "cadence"), (axs[1], t5h, "side")]:
    cats_ = [str(c) for c in t[key]]; n = len(METHODS); w = 0.8 / n
    for i, nm_ in enumerate(METHODS):
        x = np.arange(len(cats_)) + (i - (n - 1) / 2) * w
        fe = t[f"{nm_}: dollar error excl. stops"].values.astype(float)
        st_ = t[f"{nm_}: dollar error"].values.astype(float) - fe
        ax.bar(x, fe, width=w, color=MCOL[nm_], edgecolor=SURF, linewidth=1, label=nm_)
        ax.bar(x, st_, bottom=fe, width=w, color=MCOL[nm_], alpha=0.3, edgecolor=SURF, linewidth=1)
    ax.set_xticks(range(len(cats_))); ax.set_xticklabels(cats_); ax.grid(axis="x", visible=False)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
axs[0].set_title("Stream level, by cadence"); axs[1].set_title("Customer level, by side")
axs[0].set_ylabel("dollar error (pale part: stops for good)")
h, l = axs[0].get_legend_handles_labels(); fig.legend(h, l, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.05))
fig.suptitle("Monthly expected dollars: |received − expected| / received, by method", x=0.01, ha="left", y=1.03)
fig.subplots_adjust(bottom=0.2, wspace=0.2); save(fig, "10_monthly_methods.png", "dollar error of the three monthly methods, forecast error vs stops for good")

# 11 prediction intervals: coverage against what they should hold
ivs = [("Next amount: p10–p90 of history", t4a, "in p10–p90 band", "band if nothing changed"),
       ("Next amount: median ± 1.645σ", t4a, "in median ± 1.645σ (90%)", None),
       ("Next date: p10–p90 of past gaps", t3a[t3a.cadence.astype(str) != "daily"], "gap in p10–p90 of past gaps", "if nothing changed"),
       ("Month total: 90% interval", t5j, "coverage (90% nominal)", None)]
fig, axs = plt.subplots(1, 4, figsize=(14, 3.0), sharey=True)
for ax, (title, t, col, ref) in zip(axs, ivs):
    cats_ = [str(c) for c in t.cadence]
    ax.bar(range(len(cats_)), t[col].values.astype(float), color=[CAD_COLOR.get(c, NEUTRAL) for c in cats_], width=0.7)
    if ref:
        ax.scatter(range(len(cats_)), t[ref].values.astype(float), marker="_", s=500, color=INK, linewidths=2, zorder=3, label="if nothing changed")
        ax.legend(loc="lower right", fontsize=7.5)
    else:
        ax.axhline(0.9, color=INK2, linestyle="--", linewidth=1); ax.text(-0.45, 0.92, "90% nominal", ha="left", fontsize=8, color=INK2)
    ax.set_xticks(range(len(cats_))); ax.set_xticklabels(cats_, rotation=45, fontsize=8); ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0)); ax.grid(axis="x", visible=False); ax.set_title(title, fontsize=9)
axs[0].set_ylabel("share of actuals inside")
fig.suptitle("Prediction intervals: how often the actual value fell inside", x=0.01, ha="left", y=1.06)
save(fig, "11_intervals.png", "coverage of the amount, date and monthly intervals by cadence")

out(sec(f"§6 Figures: {len(SAVED)} PNG files in {os.path.abspath(FDIR)}", tbl(pd.DataFrame(SAVED))))

## §7 Housekeeping

In [ ]:
# ---- §7 Housekeeping ----------------------------------------------------------------------------
assert globals().get("_S1_OK"), "§1 did not finish: fix the problem it shows, then run all cells again"
mine = {f"sd_{SIG_SD}", f"fcp_{SIG_FC}", f"fcm_{SIG_FC}"}
others = sorted(o for o in set(ls_dirs(SCR)) - mine if o.startswith(("fcp_", "fcm_")))
sizes = pd.DataFrame([{"folder": o, "GB": size_gb(f"{SCR}/{o}")} for o in others])
if CLEAN_OTHER_SCRATCH:
    for o in others:
        delete(f"{SCR}/{o}")
cur = {"stream days (shared with pkg_payment_regularity)": size_gb(P_SD), "predictions": size_gb(P_PR), "months": size_gb(P_MO)}
for df_ in (PRS, MOS, MOS2, A):
    df_.unpersist()
if not KEEP_SCRATCH:
    delete(P_PR); delete(P_MO)
times = pd.DataFrame([{"step": k, "minutes": v} for k, v in TIMES.items()])
out(sec("§7 Scratch", tbl(pd.DataFrame([{"scratch": k, "GB": v} for k, v in cur.items()]), dec=("GB",))),
    sec("Other settings' scratch" + (" (deleted)" if CLEAN_OTHER_SCRATCH else ""), tbl(sizes, dec=("GB",))) if len(sizes) else "",
    sec("Build time", tbl(times, dec=("minutes",))) if len(times) else "",
    para("Scratch kept for reruns of §3–§6." if KEEP_SCRATCH else "Prediction scratch deleted (stream days kept for pkg_payment_regularity)."))